# 🚀 EasyPasal: Experiment E6 — vLLM Serving Benchmark on Free GPU (T5.2)

This notebook benchmarks high-throughput serving of open-weight generators (**Qwen2.5-3B-Instruct** and **Llama-3.2-1B-Instruct**) using **vLLM** (PagedAttention + continuous batching) on free GPU tiers (Google Colab T4 16GB / Kaggle T4 or P100) vs the **llama.cpp CPU baseline** established in Experiment E4.

### Experimental Protocol & Acceptance Criteria (from `docs/project/task.md`):
- **Concurrency Levels**: Sweep across $c \in \{1, 4, 16, 32\}$ simultaneous streaming clients.
- **Throughput**: Measure aggregate server tokens per second (tok/s).
- **TTFT (Time-To-First-Token)**: Measure latency to first streaming token arrival across all concurrency tiers.
- **p95 Latency**: Measure end-to-end request duration scaling (seconds).
- **Baseline Comparison**: Direct comparison against E4 quad-core CPU benchmarks (`Llama-3.2-1B-Instruct` and `Qwen2.5-3B-Instruct`).

In [ ]:
# 1. Verify GPU Hardware and CUDA Environment
import sys
from pathlib import Path

print(f"Python Version: {sys.version}")
!nvidia-smi

In [ ]:
# 2. Install Dependencies
!pip install -q vllm httpx matplotlib pandas tqdm numpy

In [ ]:
# 3. Ensure Repository Setup & Paths
import sys

if not Path("src").exists():
    !git clone https://github.com/robitalhazmi/easypasal.git
    %cd easypasal
if "." not in sys.path:
    sys.path.insert(0, ".")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# 4. Launch vLLM Background Server
import subprocess
import time

import httpx

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print(f"Starting vLLM server for {MODEL_ID} on port 8000...")
server_cmd = [
    "python3",
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    MODEL_ID,
    "--gpu-memory-utilization",
    "0.90",
    "--max-model-len",
    "4096",
    "--port",
    "8000",
    "--dtype",
    "auto",
]

server_proc = subprocess.Popen(server_cmd)

# Poll /health until server is ready
max_wait_seconds = 300
start_poll = time.time()
ready = False

while time.time() - start_poll < max_wait_seconds:
    try:
        resp = httpx.get("http://localhost:8000/health", timeout=2.0)
        if resp.status_code == 200:
            print(f"\nvLLM server is healthy and ready in {time.time() - start_poll:.1f}s!")
            ready = True
            break
    except Exception:
        print(".", end="", flush=True)
    time.sleep(5)

if not ready:
    print("\nWarning: Server did not respond within timeout. Ensure GPU is attached.")

In [ ]:
# 5. Execute Concurrency Sweeps (c in 1, 4, 16, 32)
import asyncio

from experiments.e6_vllm.run_benchmark import run_e6_benchmark

report = asyncio.run(
    run_e6_benchmark(
        base_url="http://localhost:8000/v1",
        model_name=MODEL_ID,
        concurrencies=[1, 4, 16, 32],
        num_requests_per_level=32,
        max_tokens=160,
        mock=not ready,  # Fallback to realistic mock if GPU server failed
        output_file="results/e6_vllm_benchmark.json",
        plot_dir="docs/assets",
        prompts_file="data/gold/dev.jsonl",
    )
)

In [ ]:
# 6. Render Comparative Performance Visualizations
from IPython.display import Image, display

print("Throughput vs Concurrency (vLLM GPU vs llama.cpp CPU):")
display(Image(filename="docs/assets/e6_throughput_concurrency.png"))

print("p95 Latency Scaling across Concurrency Levels:")
display(Image(filename="docs/assets/e6_latency_p95_concurrency.png"))

print("vLLM Time-To-First-Token (TTFT) across Concurrency Levels:")
display(Image(filename="docs/assets/e6_ttft_scaling.png"))

In [ ]:
# 7. Teardown Server
if "server_proc" in locals() and server_proc.poll() is None:
    server_proc.terminate()
    print("vLLM server stopped.")